# CourtVision on Colab

1. Runtime → Change runtime type → **T4 GPU**.
2. In the Secrets panel (key icon), add `GITHUB_TOKEN` (if the repo is private), `ROBOFLOW_API_KEY`, and `COURTVISION_PASSWORD`, and enable notebook access.
3. Put real rosters, tags and videos on Google Drive under `MyDrive/CourtVision/` (never in git).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CS496/courtvision'

In [ ]:
import os
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')
url = f'https://{token}@github.com/Mythventor/CourtVision.git' if token else 'https://github.com/Mythventor/CourtVision.git'
if not os.path.exists('/content/CourtVision'):
    !git clone -q {url} /content/CourtVision
else:
    !git -C /content/CourtVision pull -q
%cd /content/CourtVision
!pip install -q -e ".[cv,app]"

for name in ('ROBOFLOW_API_KEY', 'COURTVISION_PASSWORD'):
    value = secret(name)
    if value:
        os.environ[name] = value

In [ ]:
# Use a config on Drive that points at the real rosters, e.g. a copy of configs/default.yaml with
#   roster_csv: /content/drive/MyDrive/CourtVision/rosters.csv
#   output_dir: /content/drive/MyDrive/CourtVision/outputs
CONFIG = f'{DRIVE}/config.yaml'
config_arg = f'--config {CONFIG}' if os.path.exists(CONFIG) else ''
!python -m courtvision.app --share {config_arg}